# ListingLens — Exploratory Data Analysis (SQL over DuckDB)

This notebook explores the ListingLens warehouse with **SQL queries** run against a
**DuckDB** database, then visualizes the results with pandas + matplotlib.

It combines two data modalities:
- **Reviews** (single-turn) — sentiment, ratings, complaint topics.
- **Support conversations** (multi-turn) — intent, sentiment trajectory, resolution/escalation.

**Setup:** build the warehouse first from the repo root:
```bash
python -m scripts.build_duckdb
```

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

DB = Path('..') / 'data' / 'processed' / 'listinglens.duckdb'
con = duckdb.connect(str(DB), read_only=True)

def q(sql):
    return con.execute(sql).df()

q("SELECT table_name FROM information_schema.tables ORDER BY table_name")

## 1. Portfolio overview
Which products carry the most negative reviews?

In [ ]:
df = q('''
  SELECT asin, total_reviews, avg_rating, pct_negative, pct_positive
  FROM product_metrics
  ORDER BY pct_negative DESC
''')

ax = df.plot.barh(x='asin', y='pct_negative', legend=False, color='#d1495b', figsize=(8,5))
ax.set_xlabel('% negative reviews'); ax.set_ylabel('ASIN')
ax.set_title('Negative-review share by product'); ax.invert_yaxis()
plt.tight_layout(); plt.show()
df

## 2. Sentiment vs star rating
Do low ratings actually carry negative language? (Validates the sentiment model.)

In [ ]:
df = q('''
  SELECT rating,
         COUNT(*) AS n_reviews,
         ROUND(AVG(compound_score),3) AS avg_compound
  FROM reviews
  GROUP BY rating ORDER BY rating
''')

ax = df.plot.bar(x='rating', y='avg_compound', legend=False, figsize=(7,4),
                 color=['#d1495b' if v < 0 else '#00a878' for v in df['avg_compound']])
ax.axhline(0, color='#888', lw=0.8)
ax.set_xlabel('Star rating'); ax.set_ylabel('Avg compound sentiment')
ax.set_title('Sentiment by star rating'); plt.tight_layout(); plt.show()
df

## 3. Support intent mix
What customers contact support about, aggregated across the catalog.

In [ ]:
df = q('''
  SELECT intent, SUM(count) AS conversations
  FROM conversation_intents
  GROUP BY intent ORDER BY conversations DESC
''')

ax = df.plot.barh(x='intent', y='conversations', legend=False, color='#3a86ff', figsize=(8,5))
ax.set_xlabel('Conversations'); ax.set_ylabel('')
ax.set_title('Support intent distribution'); ax.invert_yaxis()
plt.tight_layout(); plt.show()
df

## 4. Conversation operations
Resolution and escalation rates by product — the operational metrics a support org tracks.

In [ ]:
df = q('''
  SELECT asin,
         COUNT(*) AS conversations,
         ROUND(AVG(CASE WHEN resolved  THEN 1 ELSE 0 END),3) AS resolution_rate,
         ROUND(AVG(CASE WHEN escalated THEN 1 ELSE 0 END),3) AS escalation_rate,
         ROUND(AVG(n_turns),2) AS avg_turns
  FROM conversations
  GROUP BY asin ORDER BY escalation_rate DESC
''')
df

## 5. Do noisier products escalate more?
Join the review signal to the conversation signal — a cross-modality correlation.

In [ ]:
df = q('''
  SELECT m.asin, m.pct_negative,
         ROUND(AVG(CASE WHEN c.escalated THEN 1 ELSE 0 END),3) AS escalation_rate
  FROM product_metrics m
  JOIN conversations c USING (asin)
  GROUP BY m.asin, m.pct_negative
  ORDER BY m.pct_negative DESC
''')

ax = df.plot.scatter(x='pct_negative', y='escalation_rate', s=60, color='#8338ec', figsize=(7,5))
ax.set_xlabel('% negative reviews'); ax.set_ylabel('Support escalation rate')
ax.set_title('Review negativity vs support escalation')
plt.tight_layout(); plt.show()
df

---
**Takeaway:** DuckDB lets us run analytical SQL directly over the precomputed review + conversation
artifacts, joining the two modalities to surface where product-quality signals predict support load.
The same warehouse backs the app's runtime loader (with a file fallback).